# Building the KSP `.cfg` Parser

KSP stores all part data in `.cfg` files — one per part, living under `GameData/Squad/Parts/`.
This notebook builds a parser for that format from scratch, then uses it to extract a clean parts library.

**Plan:**
1. Read a raw `.cfg` file and understand its structure
2. Build a line classifier to identify the different line types
3. Build a recursive block parser that turns `.cfg` text into nested Python dicts
4. Handle KSP's localization string quirk
5. Write field extractors for each part category (engines, tanks, command pods)
6. Walk the full `Squad/Parts/` directory and parse everything
7. Save the result as JSON

## Step 1 — Read a raw `.cfg` file

Let's start by just reading one file and looking at it. We'll use the Thud engine as our running example.

In [1]:
THUD_PATH = (
    "/Users/moss/Library/Application Support/Steam/steamapps/common/"
    "Kerbal Space Program/GameData/Squad/Parts/Engine/"
    "liquidEngineMk55/liquidEngineMk55.cfg"
)

with open(THUD_PATH, encoding='utf-8-sig') as f:
    raw = f.read()

# Print first 50 lines so we can see the structure
for i, line in enumerate(raw.splitlines()[:50], 1):
    print(f"{i:3d}  {line}")

  1  PART
  2  {
  3  	name = radialLiquidEngine1-2
  4  	module = Part
  5  	author = NovaSilisko, Porkjet
  6  	mesh = Thud.mu
  7  	rescaleFactor = 1
  8  	node_attach = 0.0, 0.0, 0.0, 0.0, 0.0, 1.0
  9  	fx_exhaustFlame_blue_small = 0.0, -0.5337813, 0.1355984, 0.0, 1.0, 0.0, running
 10  	fx_exhaustLight_blue = 0.0, -0.5337813, 0.1355984, 0.0, 0.0, 1.0, running
 11  	sound_vent_medium = engage
 12  	sound_rocket_hard = running
 13  	sound_vent_soft = disengage
 14  	sound_explosion_low = flameout
 15  	TechRequired = advRocketry
 16  	entryCost = 3500
 17  	cost = 820
 18  	category = Engine
 19  	subcategory = 0
 20  	title = #autoLOC_500448 //#autoLOC_500448 = Mk-55 "Thud" Liquid Fuel Engine
 21  	manufacturer = #autoLOC_501637 //#autoLOC_501637 = Rockomax Conglomerate
 22  	description = #autoLOC_500449 //#autoLOC_500449 = After an intensive search for an engineer crazy enough to plan and build a revolutionary new engine type, researchers turned to renowned engineer Eumon Kerman

## Step 2 — Understand the format

Looking at the file above, we can see the grammar is simple:

```
PART
{
    key = value         # flat key-value pair
    key = value // comment  # inline comments after //

    BLOCK_NAME          # start of a sub-block (name on its own line)
    {                   # opening brace (always on next line in stock KSP)
        key = value
        NESTED_BLOCK
        {
            key = value
        }
    }
}
```

**Line types we need to handle:**
- Empty / whitespace-only → skip
- Pure comment (`// ...`) → skip
- `KEY = VALUE` → key-value pair
- `BLOCK_NAME` (no `=`, no braces) → upcoming block name
- `{` → open a new block (using the pending block name)
- `}` → close current block

**One quirk:** `title` and other display strings use KSP's localization system:
```
title = #autoLOC_500448 //#autoLOC_500448 = Mk-55 "Thud" Liquid Fuel Engine
```
The human-readable name is hiding in the `//` comment. We'll handle this separately.

## Step 3 — Line classifier

Before building the full parser, let's write a function that identifies what kind of line we're looking at.
This makes the parser logic easy to read.

In [2]:
def classify_line(raw_line):
    """
    Given a raw line from a .cfg file, return a tuple of (line_type, content).

    Line types:
        'empty'      -> nothing here, skip it
        'open'       -> '{' — opens a new block
        'close'      -> '}' — closes current block
        'kv'         -> key-value pair, content = (key, raw_value_string)
        'block_name' -> upcoming block name, content = name string
    """
    line = raw_line.strip()

    # Empty or pure comment
    if not line or line.startswith('//'):
        return ('empty', None)

    if line == '{':
        return ('open', None)

    if line == '}':
        return ('close', None)

    if '=' in line:
        key, _, value = line.partition('=')
        # Keep the raw value — we'll clean it up later (handles localization quirk)
        return ('kv', (key.strip(), value.strip()))

    # Must be a block name (e.g., MODULE, RESOURCE, PROPELLANT)
    return ('block_name', line)

In [3]:
# Quick sanity check on the first 25 lines of the Thud cfg
for raw_line in raw.splitlines()[:25]:
    kind, content = classify_line(raw_line)
    if kind != 'empty':
        print(f"  {kind:<12}  {content}")

  block_name    PART
  open          None
  kv            ('name', 'radialLiquidEngine1-2')
  kv            ('module', 'Part')
  kv            ('author', 'NovaSilisko, Porkjet')
  kv            ('mesh', 'Thud.mu')
  kv            ('rescaleFactor', '1')
  kv            ('node_attach', '0.0, 0.0, 0.0, 0.0, 0.0, 1.0')
  kv            ('fx_exhaustFlame_blue_small', '0.0, -0.5337813, 0.1355984, 0.0, 1.0, 0.0, running')
  kv            ('fx_exhaustLight_blue', '0.0, -0.5337813, 0.1355984, 0.0, 0.0, 1.0, running')
  kv            ('sound_vent_medium', 'engage')
  kv            ('sound_rocket_hard', 'running')
  kv            ('sound_vent_soft', 'disengage')
  kv            ('sound_explosion_low', 'flameout')
  kv            ('TechRequired', 'advRocketry')
  kv            ('entryCost', '3500')
  kv            ('cost', '820')
  kv            ('category', 'Engine')
  kv            ('subcategory', '0')
  kv            ('title', '#autoLOC_500448 //#autoLOC_500448 = Mk-55 "Thud" Liquid Fuel Engine'

## Step 4 — The parser

Now we build the parser itself. The idea is a **stack machine**:

- We keep a stack of "current block" dicts.
- When we see `{`, we push a new block onto the stack.
- When we see `}`, we pop the current block and attach it to its parent.
- When we see a key-value pair, we add it to the top of the stack.

Each block is represented as a dict with:
- `_type`: the block name (e.g., `'PART'`, `'MODULE'`, `'RESOURCE'`)
- `_children`: list of sub-blocks
- All other keys: the key-value pairs found inside this block

When the same key appears more than once (e.g., multiple `key = ...` inside `atmosphereCurve`),
we collect the values into a list.

In [4]:
def parse_cfg(text):
    """
    Parse KSP .cfg format into a nested dict structure.

    Returns a root dict whose '_children' contains the top-level blocks
    (usually just one: the PART block).

    Each block looks like:
        {
            '_type': 'MODULE',
            '_children': [...],  # nested sub-blocks
            'name': 'ModuleEngines',
            'maxThrust': '120',
            ...
        }
    """
    root = {'_type': 'ROOT', '_children': []}
    stack = [root]
    pending_name = None  # block name waiting for its '{'

    for raw_line in text.splitlines():
        kind, content = classify_line(raw_line)

        if kind == 'empty':
            continue

        elif kind == 'block_name':
            pending_name = content

        elif kind == 'open':
            new_block = {'_type': pending_name or 'UNNAMED', '_children': []}
            stack[-1]['_children'].append(new_block)
            stack.append(new_block)
            pending_name = None

        elif kind == 'close':
            if len(stack) > 1:  # never pop the root
                stack.pop()

        elif kind == 'kv':
            key, value = content
            current = stack[-1]
            if key in current:
                # Same key appears more than once — collect into a list
                if not isinstance(current[key], list):
                    current[key] = [current[key]]
                current[key].append(value)
            else:
                current[key] = value

    return root

In [5]:
import json

parsed = parse_cfg(raw)

# The top-level result has a single child: the PART block
part = parsed['_children'][0]

print("Top-level block type:", part['_type'])
print("Internal name:", part.get('name'))
print("Mass:", part.get('mass'))
print("Number of sub-blocks:", len(part['_children']))
print()

# List all sub-block types
for child in part['_children']:
    module_name = child.get('name', '')
    print(f"  sub-block: {child['_type']}" + (f" ({module_name})" if module_name else ""))

Top-level block type: PART
Internal name: radialLiquidEngine1-2
Mass: 0.9
Number of sub-blocks: 6

  sub-block: MODULE (ModuleEngines)
  sub-block: MODULE (ModuleGimbal)
  sub-block: MODULE (FXModuleAnimateThrottle)
  sub-block: MODULE (ModuleTestSubject)
  sub-block: MODULE (ModuleSurfaceFX)
  sub-block: MODULE (ModuleCargoPart)


## Step 5 — Handle the localization quirk

KSP titles look like:
```
title = #autoLOC_500448 //#autoLOC_500448 = Mk-55 "Thud" Liquid Fuel Engine
```

The `#autoLOC_XXXXX` part is a localization key the game resolves at runtime.
For us, the human-readable name is everything after `= ` in the `//` comment.

Let's write a small helper to extract it.

In [6]:
import re

def clean_value(raw_value):
    """
    Clean a raw cfg value string.

    - If it's a KSP localization string (#autoLOC_...), extract the human name
      from the trailing comment.
    - Otherwise, strip any trailing // comment.
    """
    if raw_value.startswith('#autoLOC_'):
        # Pattern: #autoLOC_XXXXX //#autoLOC_XXXXX = Human Readable Name
        match = re.search(r'//\s*#autoLOC_\d+\s*=\s*(.+)', raw_value)
        if match:
            return match.group(1).strip()
        # Fallback: just return the localization key itself
        return raw_value.split()[0]
    # Strip trailing // comment if present
    if '//' in raw_value:
        return raw_value[:raw_value.index('//')].strip()
    return raw_value


# Demo: compare raw vs cleaned title
raw_title = part.get('title', '')
print("Raw title value:")
print(" ", raw_title)
print()
print("Cleaned title:")
print(" ", clean_value(raw_title))

Raw title value:
  #autoLOC_500448 //#autoLOC_500448 = Mk-55 "Thud" Liquid Fuel Engine

Cleaned title:
  Mk-55 "Thud" Liquid Fuel Engine


## Step 6 — Field extractors

Now the useful part: given a parsed part block, pull out only the fields we actually care about.

We'll write three extractors:
- `extract_engine` — thrust, Isp, propellants
- `extract_tank` — resources and their capacities
- `extract_command` — just confirms it has a command module

And a general `extract_part` that combines the common fields with the category-specific ones.

In [7]:
def get_children_of_type(block, block_type):
    """Return all direct children of a block with a given _type."""
    return [c for c in block['_children'] if c['_type'] == block_type]


def get_module(block, module_name):
    """Find a MODULE child block with a specific 'name' field."""
    for child in get_children_of_type(block, 'MODULE'):
        if child.get('name') == module_name:
            return child
    return None


def parse_atmo_curve(module_block):
    """
    Extract Isp values from an atmosphereCurve sub-block.
    Returns {'vacuum': float, 'sea_level': float} or None.

    atmosphereCurve keys look like:
        key = 0 305     <- altitude=0 (vacuum), Isp=305
        key = 1 275     <- altitude=1 (sea level), Isp=275
    """
    curve_blocks = get_children_of_type(module_block, 'atmosphereCurve')
    if not curve_blocks:
        return None
    curve = curve_blocks[0]
    keys = curve.get('key', [])
    if isinstance(keys, str):
        keys = [keys]
    isp = {}
    for entry in keys:
        parts = entry.split()
        if len(parts) >= 2:
            altitude, value = parts[0], parts[1]
            if altitude == '0':
                isp['vacuum'] = float(value)
            elif altitude == '1':
                isp['sea_level'] = float(value)
    return isp if isp else None


def extract_engine(part_block):
    """
    Extract engine-specific data from a parsed PART block.
    Returns a dict of engine fields, or None if this part has no engine module.
    """
    engine_mod = get_module(part_block, 'ModuleEngines')
    if engine_mod is None:
        engine_mod = get_module(part_block, 'ModuleEnginesFX')
    if engine_mod is None:
        return None

    # Propellants: each PROPELLANT child block has name + ratio
    propellants = {}
    for prop in get_children_of_type(engine_mod, 'PROPELLANT'):
        prop_name = prop.get('name', 'Unknown')
        prop_ratio = float(prop.get('ratio', 1.0))
        propellants[prop_name] = prop_ratio

    return {
        'max_thrust_kn': float(engine_mod.get('maxThrust', 0)),
        'min_thrust_kn': float(engine_mod.get('minThrust', 0)),
        'engine_type': engine_mod.get('EngineType', 'LiquidFuel'),
        'propellants': propellants,
        'isp': parse_atmo_curve(engine_mod),
    }


def extract_tank(part_block):
    """
    Extract fuel tank data: resource name and max capacity.
    Returns a dict of {resource_name: max_amount}, or None if no resources.
    """
    resources = {}
    for res in get_children_of_type(part_block, 'RESOURCE'):
        name = res.get('name')
        max_amt = res.get('maxAmount')
        if name and max_amt:
            # Strip inline comments from maxAmount just in case
            resources[name] = float(clean_value(max_amt))
    return resources if resources else None


def extract_nodes(part_block):
    """
    Extract attachment nodes.
    node_stack_top, node_stack_bottom, etc. → {name: {pos, dir, size}}
    node_attach → surface attachment point
    """
    nodes = {}
    for key, raw_val in part_block.items():
        if key.startswith('node_stack_') or key == 'node_attach':
            parts = clean_value(raw_val).split(',')
            if len(parts) >= 6:
                node_name = key.replace('node_stack_', '').replace('node_', '')
                try:
                    nodes[node_name] = {
                        'pos': [float(parts[0]), float(parts[1]), float(parts[2])],
                        'dir': [float(parts[3]), float(parts[4]), float(parts[5])],
                        'size': int(float(parts[6])) if len(parts) > 6 else 0,
                    }
                except ValueError:
                    pass  # skip malformed nodes
    return nodes


def extract_part(part_block):
    """
    Extract all fields we care about from a parsed PART block.
    Returns a clean dict representing one part.
    """
    # Determine category
    raw_category = clean_value(part_block.get('category', ''))

    # Check for command capability regardless of category label
    has_command = get_module(part_block, 'ModuleCommand') is not None

    part = {
        'name': part_block.get('name', ''),
        'title': clean_value(part_block.get('title', '')),
        'category': raw_category,
        'mass_t': float(clean_value(part_block.get('mass', '0'))),
        'cost': float(clean_value(part_block.get('cost', '0'))),
        'crash_tolerance': float(clean_value(part_block.get('crashTolerance', '0'))),
        'attach_rules': clean_value(part_block.get('attachRules', '')),
        'nodes': extract_nodes(part_block),
        'is_command': has_command,
        'engine': extract_engine(part_block),
        'resources': extract_tank(part_block),
    }
    return part

In [8]:
# Test on our Thud engine
thud = extract_part(part)
print(json.dumps(thud, indent=2))

{
  "name": "radialLiquidEngine1-2",
  "title": "Mk-55 \"Thud\" Liquid Fuel Engine",
  "category": "Engine",
  "mass_t": 0.9,
  "cost": 820.0,
  "crash_tolerance": 7.0,
  "attach_rules": "0,1,0,1,0",
  "nodes": {
    "attach": {
      "pos": [
        0.0,
        0.0,
        0.0
      ],
      "dir": [
        0.0,
        0.0,
        1.0
      ],
      "size": 0
    }
  },
  "is_command": false,
  "engine": {
    "max_thrust_kn": 120.0,
    "min_thrust_kn": 0.0,
    "engine_type": "LiquidFuel",
    "propellants": {
      "LiquidFuel": 0.9,
      "Oxidizer": 1.1
    },
    "isp": {
      "vacuum": 305.0,
      "sea_level": 275.0
    }
  },
  "resources": null
}


## Step 7 — Walk the full parts directory

Now let's apply everything to the entire `Squad/Parts/` directory.
We'll collect every `.cfg` file, parse it, and run `extract_part` on it.

In [9]:
from pathlib import Path

PARTS_DIR = Path(
    "/Users/moss/Library/Application Support/Steam/steamapps/common/"
    "Kerbal Space Program/GameData/Squad/Parts"
)

cfg_files = sorted(PARTS_DIR.rglob("*.cfg"))
print(f"Found {len(cfg_files)} .cfg files")

Found 359 .cfg files


In [10]:
parts_library = []
errors = []

for cfg_path in cfg_files:
    try:
        text = cfg_path.read_text(encoding='utf-8-sig', errors='replace')
        parsed = parse_cfg(text)

        # Each cfg file may have one PART block at the top level
        for child in parsed['_children']:
            if child['_type'] == 'PART':
                extracted = extract_part(child)
                extracted['_source_file'] = str(cfg_path.relative_to(PARTS_DIR))
                parts_library.append(extracted)

    except Exception as e:
        errors.append((str(cfg_path), str(e)))

print(f"Parsed {len(parts_library)} parts")
if errors:
    print(f"{len(errors)} errors:")
    for path, err in errors:
        print(f"  {path}: {err}")

Parsed 358 parts


## Step 8 — Inspect the results

Let's look at what we got — engines, tanks, and command pods.

In [11]:
# Summary by category
from collections import Counter

categories = Counter(p['category'] for p in parts_library)
for cat, count in categories.most_common():
    print(f"  {cat or '(no category)':30s}  {count}")

  Aero                            53
  FuelTank                        39
  Engine                          34
  Utility                         30
  Structural                      29
  Pods                            22
  Coupling                        19
  Electrical                      19
  Propulsion                      15
  Science                         14
  none                            13
  Ground                          13
  Cargo                           12
  Thermal                         11
  Payload                         11
  Control                         9
  Communication                   9
  (no category)                   6


In [12]:
# All engines: name, thrust, vacuum Isp, sea-level Isp
engines = [p for p in parts_library if p['engine'] is not None]
print(f"{len(engines)} engines found\n")

print(f"{'Name':<45} {'Thrust kN':>10} {'Isp vac':>9} {'Isp SL':>9}")
print("-" * 78)
for p in sorted(engines, key=lambda x: x['engine']['max_thrust_kn']):
    e = p['engine']
    isp = e.get('isp') or {}
    print(
        f"{p['title']:<45}"
        f"{e['max_thrust_kn']:>10.0f}"
        f"{isp.get('vacuum', 0):>9.0f}"
        f"{isp.get('sea_level', 0):>9.0f}"
    )

39 engines found

Name                                           Thrust kN   Isp vac    Isp SL
------------------------------------------------------------------------------
IX-6315 "Dawn" Electric Propulsion System             2     4200      100
LV-1R "Spider" Liquid Fuel Engine                     2      290      260
LV-1 "Ant" Liquid Fuel Engine                         2      315       80
FM1 "Mite" Solid Fuel Booster                        12      210      185
24-77 "Twitch" Liquid Fuel Engine                    16      290      250
24-77 "Twitch" Liquid Fuel Engine                    16      290      275
Sepratron I                                          18      154      118
O-10 "Puff" MonoPropellant Fuel Engine               20      250      120
48-7S "Spark" Liquid Fuel Engine                     20      320      265
J-20 "Juno" Basic Jet Engine                         20     6400        0
F3S0 "Shrimp" Solid Fuel Booster                     30      215      190
LV-909 "Terr

In [13]:
# All fuel tanks that hold LiquidFuel+Oxidizer (standard rocket tanks)
lfo_tanks = [
    p for p in parts_library
    if p['resources']
    and 'LiquidFuel' in p['resources']
    and 'Oxidizer' in p['resources']
]
print(f"{len(lfo_tanks)} LF+Ox tanks found\n")

print(f"{'Name':<40} {'Dry mass t':>11} {'LF':>8} {'Ox':>8}")
print("-" * 72)
for p in sorted(lfo_tanks, key=lambda x: x['resources'].get('LiquidFuel', 0)):
    r = p['resources']
    print(
        f"{p['title']:<40}"
        f"{p['mass_t']:>11.3f}"
        f"{r.get('LiquidFuel', 0):>8.0f}"
        f"{r.get('Oxidizer', 0):>8.0f}"
    )

34 LF+Ox tanks found

Name                                      Dry mass t       LF       Ox
------------------------------------------------------------------------
R-4 'Dumpling' External Tank                  0.014      10      12
Oscar-B Fuel Tank                             0.025      18      22
R-11 'Baguette' External Tank                 0.034      24      30
R-12 'Doughnut' External Tank                 0.037      27      33
MPO Probe                                     0.395      45      55
FL-T100 Fuel Tank                             0.062      45      55
FL-T200 Fuel Tank                             0.125      90     110
FL-T400 Fuel Tank                             0.250     180     220
Mk2 Bicoupler                                 0.290     180     220
Mk2 to 1.25m Adapter                          0.290     180     220
Mk2 Rocket Fuel Fuselage Short                0.290     180     220
Rockomax X200-8 Fuel Tank                     0.500     360     440
FL-T800 Fuel Tank 

In [14]:
# Command pods / probe cores
command_parts = [p for p in parts_library if p['is_command']]
print(f"{len(command_parts)} command parts found\n")

for p in sorted(command_parts, key=lambda x: x['mass_t']):
    print(f"  {p['title']:<45}  {p['mass_t']:.3f} t")

24 command parts found

  Probodobodyne OKTO2                            0.040 t
  Probodobodyne Stayputnik                       0.050 t
  Probodobodyne QBE                              0.070 t
  Probodobodyne HECS                             0.100 t
  Probodobodyne HECS                             0.100 t
  Probodobodyne OKTO                             0.100 t
  RC-001S Remote Guidance Unit                   0.100 t
  Probodobodyne RoveMate                         0.150 t
  MK2 Drone Core                                 0.200 t
  Probodobodyne HECS2                            0.200 t
  MPO Probe                                      0.395 t
  MTM Stage                                      0.415 t
  RC-L01 Remote Guidance Unit                    0.500 t
  Mk1 Lander Can                                 0.600 t
  Mk1 Command Pod                                0.800 t
  PPD-12 Cupola Module                           0.940 t
  Mk1 Inline Cockpit                             1.000 t
  Mk1 C

## Step 9 — Save to JSON

Now we save the full parts library to a JSON file that the rest of the project can load.

In [15]:
import json
from pathlib import Path

OUT_DIR = Path("/Users/moss/Desktop/Personal_documents/repos/to_the_mun/data")
OUT_DIR.mkdir(exist_ok=True)

out_path = OUT_DIR / "parts_library.json"
with open(out_path, 'w') as f:
    json.dump(parts_library, f, indent=2)

print(f"Saved {len(parts_library)} parts to {out_path}")
print(f"File size: {out_path.stat().st_size / 1024:.1f} KB")

Saved 358 parts to /Users/moss/Desktop/Personal_documents/repos/to_the_mun/data/parts_library.json
File size: 267.8 KB


In [16]:
print (f"cfg files found: {len(cfg_files)}")
from collections import Counter

top_level_blocks = Counter()
for cfg_path in cfg_files:
    text = cfg_path.read_text(encoding='utf-8-sig', errors = 'replace')
    parsed = parse_cfg(text)
    for child in parsed['_children']:
        top_level_blocks[child['_type']] += 1

print('top level blocks found across all files:')
for block_type, count in top_level_blocks.most_common():
    print(f'{block_type}: {count}')


cfg files found: 359
top level blocks found across all files:
PART: 358
VARIANTTHEME: 13
MODULE: 7
RESOURCE: 1
